# Signalwatch: false-alarm research pilot

This notebook runs the shared CPU research pipeline from a hash-verified source snapshot. Published-test comparisons are exploratory. Unknown labels remain excluded from detection denominators. The source model and published reports are preserved. No captured packets are transmitted.

Source commit: `3d43968feda32c8eb2719f28082388ed2f28fb90`; uncommitted snapshot: `False`. Mode: **public-data research pilot**.

In [ ]:
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile
BASE = pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
WORK = pathlib.Path('/tmp/signalwatch') if pathlib.Path('/kaggle/working').exists() else BASE / '.signalwatch'
WORK.mkdir(parents=True, exist_ok=True)
PROJECT = WORK / 'signalwatch-source'
PAYLOAD = '...'
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == '4c0756b624542684eed80ef45a020f86e2c1dda125d4026abb2d34bab7b8524c'
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as bundle:
    assert all(not pathlib.PurePosixPath(p).is_absolute() and '..' not in pathlib.PurePosixPath(p).parts for p in bundle.namelist())
    bundle.extractall(PROJECT)
snapshot = json.loads((PROJECT / 'RESEARCH_SOURCE.json').read_text())
for filename, expected in snapshot['files'].items():
    assert hashlib.sha256((PROJECT / filename).read_bytes()).hexdigest() == expected
print('Verified source snapshot:', snapshot['commit'])

## Pinned offline environment
The attached private inputs supply Python 3.12, dependency wheels and packet captures. All archives are hash-verified before extraction. No external downloads are needed during execution.

In [ ]:
import tarfile
EXPECTED_INPUT_SHA = '4e5d22b92d452b9026b7bd4723752ae02f6658fabff6cac2091ddf9dc1d59f38'
candidates = [p for p in pathlib.Path('/kaggle/input').rglob('input_manifest.json') if hashlib.sha256(p.read_bytes()).hexdigest() == EXPECTED_INPUT_SHA]
assert len(candidates) == 1, 'Attach the exact private input dataset version used to build this notebook.'
INPUT = candidates[0].parent
inputs = json.loads(candidates[0].read_text())
OUTPUT = WORK / 'signalwatch-results'
OUTPUT.mkdir(parents=True, exist_ok=True)
for name, details in inputs['archives'].items():
    path = INPUT / name
    with path.open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == details['sha256']
    with tarfile.open(path) as archive:
        archive.extractall(OUTPUT if name == 'captures.tar.gz' else WORK, filter='data')
PYTHON = WORK / 'python/bin/python3.12'
subprocess.run([str(PYTHON), '-m', 'ensurepip', '--upgrade'], check=True, timeout=120)
subprocess.run([str(PYTHON), '-m', 'pip', 'install', '--quiet', '--no-index', '--find-links', str(WORK / 'wheels'), '-r', str(PROJECT / 'requirements-research.txt')], check=True, timeout=1200)
RUN_ENV = dict(os.environ, MPLCONFIGDIR=str(WORK / 'matplotlib-cache'), XDG_CACHE_HOME=str(WORK / 'cache'), OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2')
def run_stage(stage):
    command = [str(PYTHON), '-m', 'research', stage, '--output', str(OUTPUT)] + []
    subprocess.run(command, cwd=PROJECT, env=RUN_ENV, check=True, timeout=14400)


## Prepare data and reproduce the baseline
Capture and model checkpoints can be reused only with matching code, data and dependency fingerprints.

In [ ]:
run_stage('prepare')
run_stage('baseline')

## Compare detectors and validation-selected operating points
Five model seeds are used in the full pilot. The prospective policy is RF seed 42 at a 0.1% validation false-positive budget; published-test results do not choose that policy.

In [ ]:
run_stage('compare')
run_stage('report')

## Evaluate the frozen policy on fresh captures
Only runs after the baseline and comparison have frozen the candidate. Do not retune after viewing these results.

In [ ]:
run_stage('evaluate-frozen')
run_stage('report')

## Downloadable evidence
The archive includes manifests, measured metrics, all-window prediction exports, plots and findings. No API tokens, credential files, or dataset raw captures are included. Software checks and a successful run do not establish research novelty.

In [ ]:
import shutil
subprocess.run([str(PYTHON), '-m', 'pytest', '-q'], cwd=PROJECT, env=RUN_ENV, check=True, timeout=600)
from IPython.display import Markdown, display
display(Markdown((OUTPUT / 'FINDINGS.md').read_text()))
export_dir = BASE / 'signalwatch-evidence'
export_dir.mkdir(exist_ok=True)
for path in OUTPUT.iterdir():
    if path.name in ('raw', 'data', 'fresh_data'):
        continue
    if path.is_dir():
        shutil.copytree(path, export_dir / path.name, dirs_exist_ok=True)
    else:
        shutil.copy2(path, export_dir / path.name)
print(shutil.make_archive(str(BASE / 'signalwatch-evidence'), 'zip', export_dir))